# 03 — Explore `urdu-instruct` Dataset

**Purpose:** Before committing to this as our primary data source, we need to answer:

1. What does it actually look like? (columns, format)
2. Is the Urdu natural or does it read like translated text?
3. What categories/topics does it cover?
4. Are there quality issues (duplicates, very short/long, garbled text)?
5. How does it compare to Aya Urdu in quality?

**Why this matters:** This dataset will be ~90% of your training data. If it's bad, your model will be bad. 10 minutes of inspection now saves days of debugging later.

**Dataset:** `large-traversaal/urdu-instruct` — 51k GPT-4o-generated Urdu instruction-response pairs, built by Traversaal.AI (Pakistani AI company) using a modified self-instruction framework.

In [1]:
!pip install datasets pandas -q

---
## Step 1: Load and inspect structure

First question: what columns does it have? We need to know how to map it to Alpaca format (`instruction`, `input`, `output`).

In [2]:
from datasets import load_dataset
import pandas as pd

print("Loading urdu-instruct dataset...")
ds = load_dataset("large-traversaal/urdu-instruct")

# Show all available splits (train, test, validation?)
print(f"\nAvailable splits: {list(ds.keys())}")
for split_name, split_data in ds.items():
    print(f"  {split_name}: {len(split_data):,} examples")

# Pick the train split for exploration
train = ds["train"] if "train" in ds else ds[list(ds.keys())[0]]

print(f"\nColumns: {train.column_names}")
print(f"\nFirst example (raw):")
print(train[0])

Loading urdu-instruct dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/7.45M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/147k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51686 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1084 [00:00<?, ? examples/s]


Available splits: ['train', 'test']
  train: 51,686 examples
  test: 1,084 examples

Columns: ['instruction', 'output', 'category']

First example (raw):
{'instruction': 'صبح کی سیر کے فوائد پر مضمون لکھیں۔', 'output': 'صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا ہے۔', 'category': 'generation'}


In [3]:
# Show first 3 examples in detail — read the Urdu carefully
for i in range(3):
    ex = train[i]
    print(f"{'='*60}")
    print(f"Example {i+1}")
    print(f"{'='*60}")
    for col in train.column_names:
        val = str(ex[col])
        if len(val) > 500:
            val = val[:500] + "..."
        print(f"\n[{col}]:\n{val}")
    print()

Example 1

[instruction]:
صبح کی سیر کے فوائد پر مضمون لکھیں۔

[output]:
صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا ہے۔

[category]:
generation

Example 2

[instruction]:
Translate this to English: 'کتابیں ہمارے بہترین دوست ہیں۔'

[output]:
Books are our best friends.

[category]:
translation

Example 3

[instruction]:
علامہ اقبال نے اپنی شاعری میں کس فلسفے کی ترویج کی؟

[output]:
علامہ اقبال نے اپنی شاعری میں خودی اور اسلامی احیاء کے فلسفے کی ترویج کی۔

[category]:
qa



---
## Step 2: Understand the column mapping

We need to figure out which columns map to our Alpaca format:
- `instruction` → the user's request
- `input` → optional context (often empty)
- `output` → the model's response

Common column names in instruction datasets: `instruction`/`prompt`/`question` and `output`/`response`/`answer`.

In [4]:
# Show all column names and the first value of each
print("Column → first value:")
print("-" * 60)
for col in train.column_names:
    val = str(train[0][col])[:150]
    print(f"{col:25s} → {val}")

Column → first value:
------------------------------------------------------------
instruction               → صبح کی سیر کے فوائد پر مضمون لکھیں۔
output                    → صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ 
category                  → generation


---
## Step 3: Quality check — is the Urdu natural?

This is the most important check. Read 10 random examples and ask yourself:
- Does this read like natural Urdu?
- Or does it feel like translated English?
- Are there weird artifacts (random English words, broken grammar, repeated phrases)?

**You are the quality check here.** As a native Urdu speaker, your gut reaction matters more than any automated metric.

In [5]:
import random
random.seed(123)

sample_indices = random.sample(range(len(train)), 10)

for i, idx in enumerate(sample_indices):
    ex = train[idx]
    print(f"\n{'='*60}")
    print(f"Random example {i+1} (index {idx})")
    print(f"{'='*60}")
    for col in train.column_names:
        val = str(ex[col])
        if len(val) > 400:
            val = val[:400] + "..."
        print(f"\n[{col}]:\n{val}")


Random example 1 (index 3431)

[instruction]:
کسی بھی ایک موسمیات یا موسمی میلے کی تصویری وضاحت کریں۔

[output]:
بسنت کا میلہ پاکستان میں بڑی دھوم دھام سے منایا جاتا ہے۔ ہر جانب رنگ برنگی پتنگیں فلک پر اڑتی ہوئی نظر آتی ہیں اور فضا میں محبت و خوشی کا سماں ہوتا ہے۔

[category]:
generation

Random example 2 (index 17542)

[instruction]:
اگر ایک گاہک نے 500 روپے کا سامان خریدا اور 15% چھوٹ ملی، تو کتنے پیسے ڈسکاؤنٹ میں ملے؟ a) 50 b) 75 c) 100

[output]:
Reasoning: ڈسکاؤنٹ = (شرح / 100) × کل قیمت،
(15 / 100) × 500 = 75
Answer: b) 75

[category]:
reasoning

Random example 3 (index 5713)

[instruction]:
دیے گئے جملے میں موجود جذبات کی نوعیت کی شناخت کریں: 'میرا بہترین دوست میری مدد کو آیا۔' - سکون، خوشی، احسان مندی

[output]:
جملے میں جذبات کی نوعیت 'احسان مندی' ہے کیونکہ مدد کرنے پر شکرگزاری کا اظہار ہو رہا ہے۔

[category]:
sentiment

Random example 4 (index 50394)

[instruction]:
ذیل کے بیان کو 'صحت'، 'تفریح'، 'تعلیمی' یا 'ادب' کے زمرے میں تقسیم کریں: 'یہ فلم ہمیں بچوں کی ذہنی صحت کے متعل

---
## Step 4: Category distribution

The dataset description says it covers 7 Urdu task categories. Let's see if there's a category column and how balanced it is. If one category dominates, the model will be biased toward that type of task.

In [6]:
from collections import Counter

# Check for category-like columns
for col in train.column_names:
    unique_count = len(set(train[col]))
    # Category columns typically have few unique values
    if unique_count < 50:
        print(f"\nColumn '{col}' has {unique_count} unique values:")
        counts = Counter(train[col])
        for val, count in counts.most_common():
            bar = '█' * (count // (len(train) // 50))  # Simple bar chart
            print(f"  {str(val):30s} {count:>6,} ({100*count/len(train):5.1f}%) {bar}")


Column 'category' has 7 unique values:
  translation                    10,001 ( 19.3%) █████████
  reasoning                       9,590 ( 18.6%) █████████
  ethics                          9,002 ( 17.4%) ████████
  qa                              8,177 ( 15.8%) ███████
  generation                      5,907 ( 11.4%) █████
  classification                  4,662 (  9.0%) ████
  sentiment                       4,347 (  8.4%) ████


---
## Step 5: Length analysis

Checking for:
- **Too short** (<20 chars) — probably garbage or placeholder
- **Too long** (>5000 chars) — might exceed 2048 tokens, will be truncated
- **Suspiciously uniform** — if all outputs are the same length, it's a red flag for template-generated data

In [7]:
# Length stats per column
for col in train.column_names:
    # Skip non-text columns
    if not isinstance(train[0][col], str):
        continue
    
    col_lengths = [len(str(ex[col])) for ex in train]
    
    print(f"\n'{col}' character lengths:")
    print(f"  Min:    {min(col_lengths):,}")
    print(f"  Max:    {max(col_lengths):,}")
    print(f"  Mean:   {sum(col_lengths)/len(col_lengths):,.0f}")
    print(f"  Median: {sorted(col_lengths)[len(col_lengths)//2]:,}")
    print(f"  Empty:  {sum(1 for l in col_lengths if l == 0)}")
    print(f"  <20:    {sum(1 for l in col_lengths if l < 20)}")
    print(f"  >5000:  {sum(1 for l in col_lengths if l > 5000)}")


'instruction' character lengths:
  Min:    9
  Max:    269
  Mean:   72
  Median: 64
  Empty:  0
  <20:    199
  >5000:  0

'output' character lengths:
  Min:    1
  Max:    651
  Mean:   87
  Median: 82
  Empty:  0
  <20:    2598
  >5000:  0

'category' character lengths:
  Min:    2
  Max:    14
  Mean:   8
  Median: 9
  Empty:  0
  <20:    51686
  >5000:  0


---
## Step 6: Duplicate check

Duplicates are common in synthetic datasets. If the same instruction appears multiple times, the model memorizes it instead of learning the pattern. Let's check.

In [8]:
# Check for exact duplicates in the instruction/prompt column
# Use whichever column holds the instruction — adjust name as needed
text_cols = [c for c in train.column_names if isinstance(train[0][c], str)]
print(f"Text columns: {text_cols}\n")

for col in text_cols:
    values = [str(ex[col]) for ex in train]
    unique = len(set(values))
    dupes = len(values) - unique
    print(f"'{col}': {len(values):,} total, {unique:,} unique, {dupes:,} duplicates ({100*dupes/len(values):.1f}%)")

Text columns: ['instruction', 'output', 'category']

'instruction': 51,686 total, 51,686 unique, 0 duplicates (0.0%)
'output': 51,686 total, 49,289 unique, 2,397 duplicates (4.6%)
'category': 51,686 total, 7 unique, 51,679 duplicates (100.0%)


---
## Step 7: Script detection — is it actually Urdu?

Sometimes "Urdu" datasets contain mostly English, Roman Urdu, or mixed text. Urdu script uses Arabic-derived characters (Unicode range 0600–06FF). Let's check what percentage of the text is actually in Urdu script.

In [9]:
import re

def urdu_script_ratio(text):
    """What fraction of alphabetic characters are Urdu/Arabic script?"""
    if not text:
        return 0.0
    # Urdu uses Arabic script block (0600-06FF) + Arabic Supplement
    urdu_chars = len(re.findall(r'[\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]', text))
    # Total alphabetic characters (excluding digits, spaces, punctuation)
    alpha_chars = len(re.findall(r'[a-zA-Z\u0600-\u06FF\u0750-\u077F\uFB50-\uFDFF\uFE70-\uFEFF]', text))
    if alpha_chars == 0:
        return 0.0
    return urdu_chars / alpha_chars

# Check script ratio on the main text column (adjust column name as needed)
# We'll check the first text column that's likely the output/response
main_col = text_cols[-1]  # Often the last text column is the response
print(f"Checking Urdu script ratio on column: '{main_col}'")

ratios = [urdu_script_ratio(str(ex[main_col])) for ex in train]

print(f"\nUrdu script ratio in '{main_col}':")
print(f"  >90% Urdu script: {sum(1 for r in ratios if r > 0.9):,} examples")
print(f"  50-90% (mixed):   {sum(1 for r in ratios if 0.5 <= r <= 0.9):,} examples")
print(f"  <50% (mostly non-Urdu): {sum(1 for r in ratios if r < 0.5):,} examples")
print(f"  Average ratio:    {sum(ratios)/len(ratios):.1%}")

Checking Urdu script ratio on column: 'category'

Urdu script ratio in 'category':
  >90% Urdu script: 0 examples
  50-90% (mixed):   0 examples
  <50% (mostly non-Urdu): 51,686 examples
  Average ratio:    0.0%


In [10]:
# Show examples with LOW Urdu ratio — these might be English-heavy or problematic
low_urdu = [(i, ratios[i]) for i in range(len(ratios)) if ratios[i] < 0.5]

if low_urdu:
    print(f"Found {len(low_urdu)} examples with <50% Urdu script. Showing first 3:\n")
    for idx, ratio in low_urdu[:3]:
        ex = train[idx]
        print(f"--- Index {idx} (Urdu ratio: {ratio:.1%}) ---")
        for col in text_cols:
            print(f"[{col}]: {str(ex[col])[:200]}")
        print()
else:
    print("All examples have >50% Urdu script. Good sign!")

Found 51686 examples with <50% Urdu script. Showing first 3:

--- Index 0 (Urdu ratio: 0.0%) ---
[instruction]: صبح کی سیر کے فوائد پر مضمون لکھیں۔
[output]: صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا
[category]: generation

--- Index 1 (Urdu ratio: 0.0%) ---
[instruction]: Translate this to English: 'کتابیں ہمارے بہترین دوست ہیں۔'
[output]: Books are our best friends.
[category]: translation

--- Index 2 (Urdu ratio: 0.0%) ---
[instruction]: علامہ اقبال نے اپنی شاعری میں کس فلسفے کی ترویج کی؟
[output]: علامہ اقبال نے اپنی شاعری میں خودی اور اسلامی احیاء کے فلسفے کی ترویج کی۔
[category]: qa



---
## Step 8: Compare with Aya Urdu

Side-by-side comparison: pick a similar topic from both datasets and see how the Urdu reads. This tells you if `urdu-instruct` quality is close to human-written Aya.

In [11]:
# Load Aya Urdu for comparison
print("Loading Aya dataset for comparison...")
aya = load_dataset("CohereForAI/aya_dataset", split="train")

# Filter to Urdu
lang_col = None
for candidate in ['language', 'language_code', 'lang']:
    if candidate in aya.column_names:
        lang_col = candidate
        break

aya_urdu = aya.filter(lambda ex: 'urdu' in ex[lang_col].lower() or ex[lang_col] == 'urd')
print(f"Aya Urdu examples: {len(aya_urdu):,}")
print(f"Aya columns: {aya_urdu.column_names}")

Loading Aya dataset for comparison...


Filter:   0%|          | 0/202362 [00:00<?, ? examples/s]

Aya Urdu examples: 654
Aya columns: ['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']


In [12]:
# Show 3 Aya Urdu examples for comparison
print("AYA URDU (human-written):")
print("=" * 60)
for i in range(min(3, len(aya_urdu))):
    ex = aya_urdu[i]
    print(f"\nExample {i+1}:")
    for col in aya_urdu.column_names:
        val = str(ex[col])
        if len(val) > 300:
            val = val[:300] + "..."
        print(f"  [{col}]: {val}")
    print("-" * 40)

AYA URDU (human-written):

Example 1:
  [inputs]: نوری سال کے بارے میں کوی شعر بتایں
  [targets]:  نوری سال کے بارے میں اشتیاق احمد کا شعر ہے:

ر و شنی سب سے تیز چلتی ہے
اس کے جیسے نہیں کسی کی چال
ایک برس میں جو سفر کرتی ہے
اس کو کہتے ہے ایک نوری سال۔
  [language]: Urdu
  [language_code]: urd
  [annotation_type]: re-annotations
  [user_id]: 634e3e0931cf859c8d8add6e8a5cc59524f2f996b15f83ee0a150ebe82324a15
----------------------------------------

Example 2:
  [inputs]: چار دوست—ایلس، باب، کیرول اور ڈیو—ایک قطار میں بیٹھے ہیں۔ ایلس کہتی ہے، "باب بہت بائیں طرف نہیں ہے۔" باب کہتے ہیں، "ڈیو بہت دائیں طرف نہیں ہے۔" کیرول کا کہنا ہے، "ایلس بہت بائیں یا بہت دائیں طرف نہیں ہے." ڈیو کہتے ہیں، "باب بائیں سے دوسرے نمبر پر ہے۔" ہر شخص کہاں بیٹھا ہے؟
  [targets]: ایلس بہت بائیں طرف ہے، باب بائیں سے دوسرے، کیرول دائیں سے دوسرے، اور ڈیو بہت دائیں طرف ہے۔ اگر ایلس بائیں جانب نہ ہوتی تو باب کا بیان غلط ہوگا۔ اگر باب بہت بائیں طرف ہوتے تو ڈیو کا بیان غلط ہوگا۔ لہذا، ایلس بہت بائیں طرف ہے، اور ڈیو بہت دائ

In [13]:
# Show 3 urdu-instruct examples for comparison
print("URDU-INSTRUCT (GPT-4o generated):")
print("=" * 60)
for i in range(3):
    ex = train[i]
    print(f"\nExample {i+1}:")
    for col in train.column_names:
        val = str(ex[col])
        if len(val) > 300:
            val = val[:300] + "..."
        print(f"  [{col}]: {val}")
    print("-" * 40)

URDU-INSTRUCT (GPT-4o generated):

Example 1:
  [instruction]: صبح کی سیر کے فوائد پر مضمون لکھیں۔
  [output]: صبح کی سیر انسان کی صحت کے لئے بہت فائدہ مند ہے۔ یہ موٹاپا کم کرتی ہے، دماغ کو تروتازہ کرتی ہے، اور دل کی بیماریوں کے خطرے کو کم کرتی ہے۔ صبح کے تازہ ہوا میں سانس لینے سے انسان کا موڈ بھی خوشگوار رہتا ہے۔
  [category]: generation
----------------------------------------

Example 2:
  [instruction]: Translate this to English: 'کتابیں ہمارے بہترین دوست ہیں۔'
  [output]: Books are our best friends.
  [category]: translation
----------------------------------------

Example 3:
  [instruction]: علامہ اقبال نے اپنی شاعری میں کس فلسفے کی ترویج کی؟
  [output]: علامہ اقبال نے اپنی شاعری میں خودی اور اسلامی احیاء کے فلسفے کی ترویج کی۔
  [category]: qa
----------------------------------------


---
## Summary — your verdict

After running all cells, answer these questions for yourself:

1. **Does `urdu-instruct` Urdu read naturally?** (Compare with Aya)
2. **Are there obvious quality issues?** (Duplicates, English leakage, garbled text)
3. **Is the category distribution reasonable?** (Or is one topic 80% of the data?)
4. **How much cleaning is needed?** (Can you use it mostly as-is, or heavy filtering?)

### If quality is GOOD:
Use `urdu-instruct` as your primary dataset. Combine with Aya Urdu + your hand-crafted examples. Skip Alpaca translation entirely.

### If quality is MEDIOCRE:
Use the better examples from `urdu-instruct` (filter aggressively) + Aya + supplement with LLM-generated Urdu + hand-crafted.

### If quality is BAD:
Fall back to careful Alpaca translation using Claude/GPT-4 with native-style prompting.